In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.master(f"local[4]").appName("a5").getOrCreate()

26/11/06 03:50:22 WARN Utils: Your hostname, MGLSJ-EBUltra resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/06 03:50:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/06 03:50:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
sc = spark.sparkContext

In [4]:
lines = sc.textFile("../test_files/sample_hpc_rag.log")

In [13]:
def to_edge(line):
    try:
        timestamp, level, p, r = line.strip().split("\t")
        if level == "WAIT":
            return [((p, r), timestamp)]
        elif level == "HOLD":
            return [((r, p), timestamp)]
    except ValueError:
        pass
    return []

In [23]:
timed_edges = lines.flatMap(to_edge).reduceByKey(lambda x, y: min(x, y))
edges = timed_edges.map(lambda e: e[0])

print(edges.count())

18


In [24]:
procs = edges.map(lambda e: e[0] if e[0].startswith("P") else e[1]).distinct()
procs.count()

9

In [25]:
res =  edges.map(lambda e: e[1] if e[1].startswith("R") else e[0]).distinct()
res.count()

9

## Q4

In [26]:
in_degree = edges.map(lambda e: (e[1], 1)).reduceByKey(lambda x, y: x + y)

in_res = in_degree.filter(lambda n: n[0].startswith("R"))
in_res.takeOrdered(5, key=lambda n: (-n[1], n[0]))

[('R001', 2), ('R008', 2), ('R002', 1), ('R003', 1), ('R004', 1)]

In [27]:
out_degree = edges.map(lambda e: (e[0], 1)).reduceByKey(lambda x, y: x + y)

out_procs = out_degree.filter(lambda n: n[0].startswith("P"))
out_procs.takeOrdered(5, key=lambda n: (-n[1], n[0]))

[('P104', 2), ('P109', 2), ('P101', 1), ('P102', 1), ('P103', 1)]

## Q2

In [11]:
paths = procs.map(lambda proccess: (proccess,))
cycles = sc.parallelize([]) 

edges_keyed = edges.map(lambda e: (e[0], e[1] ))

for i in range(25):
    paths_keyed = paths.map(lambda path: (path[-1], path))
    joined = paths_keyed.join(edges_keyed)
    paths = joined.map(lambda item: item[1][0] + (item[1][1],) )

    detected_cycles = paths.filter(lambda path: path[0] == path[-1])
    cycles = cycles.union(detected_cycles)

    paths = paths.filter(lambda path: path[-1] not in path[:-1])

cycles.collect()

[('P107', 'R007', 'P105', 'R003', 'P107'),
 ('P105', 'R003', 'P107', 'R007', 'P105'),
 ('P106', 'R006', 'P101', 'R002', 'P106'),
 ('P101', 'R002', 'P106', 'R006', 'P101'),
 ('P103', 'R009', 'P108', 'R004', 'P102', 'R005', 'P103'),
 ('P108', 'R004', 'P102', 'R005', 'P103', 'R009', 'P108'),
 ('P102', 'R005', 'P103', 'R009', 'P108', 'R004', 'P102')]

In [12]:
def canonical(path):
    path = path[:-1]
    min_ele = min(path)
    i = path.index(min_ele)
    return path[i:] + path[:i]

cycles = cycles.map(canonical).distinct()
sorted_cycles = sorted(cycles.collect(), key = lambda c: (len(c), c))

procs_in_cycle = set()

print(len(sorted_cycles))

for cycle in sorted_cycles:
    print(*cycle)
    for ele in cycle:
        if ele.startswith("P"):
            procs_in_cycle.add(ele)
            
print(len(procs_in_cycle))
print(*sorted(list(procs_in_cycle)))


[Stage 40:==================================================>   (126 + 4) / 134]

3
P101 R002 P106 R006
P105 R003 P107 R007
P102 R005 P103 R009 P108 R004
7
P101 P102 P103 P105 P106 P107 P108


In [32]:
def cycle_edges(cycle):
    return list(zip(cycle, cycle[1:] + cycle[:1]))

cycle_edge_pairs = cycles.flatMap(lambda c: [(e, c) for e in cycle_edges(c)])
cycle_edge_times = cycle_edge_pairs.join(timed_edges)
cycle_times = cycle_edge_times.map(lambda item: item[1]).reduceByKey(lambda x, y: max(x, y))
day_count = cycle_times.map(lambda item: (item[1].split()[0], 1)).reduceByKey(lambda x, y: x+y)

[Stage 143:=============================================>       (118 + 4) / 136]

('2025-11-02', 2)
('2025-10-31', 1)


In [34]:
for pair in sorted(day_count.collect()):
    print(*pair)

[Stage 179:=========================================>           (106 + 4) / 136]

2025-10-31 1
2025-11-02 2
